[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy!

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible.

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state.

To tackle this problem, we’ve built [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we'll use Chat Models, which take a sequence of messages as input and return messages as output. LangChain supports many models via [third-party integrations](https://docs.langchain.com/oss/python/integrations/chat). This branch uses [Ollama](https://ollama.com/) for local LLMs, so **no OpenAI API key is required**.

## Using Ollama Instead of OpenAI

This notebook has been modified to use **Ollama** - a local LLM runtime - instead of OpenAI. 

### Prerequisites:
1. Ollama must be installed and running on your machine
2. You need to have at least one model pulled (e.g., `llama3.2`, `llama3.1`, `mistral`, etc.)

### Quick Setup:
```bash
# Check if Ollama is running
ollama list

# Pull a model if needed (recommended: llama3.2)
ollama pull llama3.2
```

This approach gives you:
- ✅ No API costs
- ✅ Complete privacy (runs locally)
- ✅ No API key required
- ✅ Works offline

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_ollama langchain_core langchain_community langchain-tavily

# No API key needed for Ollama. Using local LLMs.
print("Using Ollama - no API key required.")

In [2]:
# Check if Ollama is running and show available models
import subprocess
import sys

try:
    result = subprocess.run(['ollama', 'list'], capture_output=True, text=True, timeout=5)
    if result.returncode == 0:
        print("✅ Ollama is running!\n")
        print("Available models:")
        print(result.stdout)
    else:
        print("⚠️  Ollama command failed. Make sure Ollama is installed and running.")
        print("Visit: https://ollama.ai to install Ollama")
except FileNotFoundError:
    print("❌ Ollama not found. Please install Ollama from https://ollama.ai")
    sys.exit(1)
except subprocess.TimeoutExpired:
    print("⚠️  Ollama command timed out. Make sure Ollama service is running.")

✅ Ollama is running!

Available models:
NAME                  ID              SIZE      MODIFIED       
llama3:latest         365c0bd3c000    4.7 GB    34 minutes ago    
gemma3:latest         a2af6cc3eb7f    3.3 GB    7 months ago      
devstral:latest       c4b2fa0c33d7    14 GB     7 months ago      
deepseek-r1:latest    6995872bfe4c    5.2 GB    7 months ago      



In [3]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

# No API key needed for local Ollama
print("Using local Ollama - no API key required")

Using local Ollama - no API key required


[Here](https://docs.langchain.com/oss/python/langchain/models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. 

With Ollama installed locally, we can use the `langchain-ollama` package to instantiate our `ChatOllama` model object. This allows us to run LLMs completely locally without any API costs or internet connection.

Popular Ollama models you can use:
- `llama3.2` - Latest Llama model, great balance of performance and quality
- `llama3.1` - Powerful model for complex tasks
- `mistral` - Fast and efficient
- `codellama` - Optimized for coding tasks

To pull a new model: `ollama pull <model-name>`

There are [a few standard parameters](https://docs.langchain.com/oss/python/langchain/models#parameters) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses.

In [4]:
from langchain_ollama import ChatOllama

# Initialize Ollama models using llama3 (available locally)
# Using llama3:latest as the primary model
gpt4o_chat = ChatOllama(model="llama3:latest", temperature=0)
gpt35_chat = ChatOllama(model="llama3:latest", temperature=0)

# You can also try other models you have installed:
# - gemma3:latest
# - devstral:latest  
# - deepseek-r1:latest

# To see available models, run: ollama list

Chat models in LangChain have a number of [default methods](https://reference.langchain.com/python/langchain_core/runnables). For the most part, we'll be using:

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream): stream back chunks of the response
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): call the chain on an input

And, as mentioned, chat models take [messages](https://docs.langchain.com/oss/python/langchain/messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [5]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
gpt4o_chat.invoke(messages)

AIMessage(content="Hello there! It's great to meet you. Is there something I can help you with, or would you like to chat about something in particular?", additional_kwargs={}, response_metadata={'model': 'llama3:latest', 'created_at': '2026-02-01T15:58:09.059158Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1811431292, 'load_duration': 27764000, 'prompt_eval_count': 12, 'prompt_eval_duration': 270975500, 'eval_count': 31, 'eval_duration': 1512312458, 'logprobs': None, 'model_name': 'llama3:latest', 'model_provider': 'ollama'}, id='lc_run--019c19ec-fb8c-7ba3-af17-5c06f0d98190-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 31, 'total_tokens': 43})

We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.


In [6]:
gpt4o_chat.invoke("hello world")

AIMessage(content='Hello there! It\'s great to see you! "Hello World" is a classic greeting, and I\'m happy to respond in kind. How are you today?', additional_kwargs={}, response_metadata={'model': 'llama3:latest', 'created_at': '2026-02-01T15:58:11.113653Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2044823625, 'load_duration': 12703917, 'prompt_eval_count': 12, 'prompt_eval_duration': 237139208, 'eval_count': 34, 'eval_duration': 1794583417, 'logprobs': None, 'model_name': 'llama3:latest', 'model_provider': 'ollama'}, id='lc_run--019c19ed-02ab-7421-aa83-7b0a8b2bff01-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 34, 'total_tokens': 46})

In [7]:
gpt35_chat.invoke("hello world")

AIMessage(content='Hello there! It\'s great to see you! "Hello World" is a classic greeting, and I\'m happy to respond in kind. How are you today?', additional_kwargs={}, response_metadata={'model': 'llama3:latest', 'created_at': '2026-02-01T15:58:12.952009Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1826095042, 'load_duration': 16860500, 'prompt_eval_count': 12, 'prompt_eval_duration': 47737833, 'eval_count': 34, 'eval_duration': 1756225084, 'logprobs': None, 'model_name': 'llama3:latest', 'model_provider': 'ollama'}, id='lc_run--019c19ed-0ab3-7f43-9c28-e844d2910750-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 34, 'total_tokens': 46})

The interface is consistent across all chat models and models are typically initialized once at the start up each notebooks. 

So, you can easily switch between models without changing the downstream code if you have strong preference for another provider.


## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [8]:
_set_env("TAVILY_API_KEY")

In [9]:
from langchain_tavily import TavilySearch  # updated at 1.0

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "What is LangGraph?"})
search_docs = data.get("results", data)

In [10]:
search_docs

[{'url': 'https://www.geeksforgeeks.org/machine-learning/what-is-langgraph/',
  'title': 'What is LangGraph? - GeeksforGeeks',
  'content': 'LangGraph is an open-source framework built by LangChain that streamlines the creation and management of AI agent workflows.',
  'score': 0.9999926,
  'raw_content': None},
 {'url': 'https://www.datacamp.com/tutorial/langgraph-tutorial',
  'title': 'LangGraph Tutorial: What Is LangGraph and How to Use It?',
  'content': 'LangGraph is a library within the LangChain ecosystem that provides a framework for defining, coordinating, and executing multiple LLM agents (or chains) in a structured and efficient manner. By managing the flow of data and the sequence of operations, LangGraph allows developers to focus on the high-level logic of their applications rather than the intricacies of agent coordination. Whether you need a chatbot that can handle various types of user requests or a multi-agent system that performs complex tasks, LangGraph provides the